In [3]:
from copy import deepcopy

import torch 
import torch.nn.functional as F
from torch import nn
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, AutoModel

In [4]:
model_path = "G:/model_weights/huggingface_model/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [19]:

config = AutoConfig.for_model('llama')
config.hidden_size = 24
config.intermediate_size = config.hidden_size * 4
config.num_attention_heads = 4
config.num_hidden_layers = 4
config.num_key_value_heads = 2
config.vocab_size = 128

In [21]:
model = AutoModel.from_config(config)  # 没带因果头

In [5]:
print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [6]:
for name, _ in model.named_children():
    print("name:",name, type(_))
    print('-----')
    print("model:",_)

name: model <class 'transformers.models.qwen3.modeling_qwen3.Qwen3Model'>
-----
model: Qwen3Model(
  (embed_tokens): Embedding(151936, 2560)
  (layers): ModuleList(
    (0-35): 36 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
        (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
        (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
        (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
        (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
      (

In [7]:
for name, child in model.named_children():
    print(name)
    # for n, child2 in child.named_children():
    #     print(n)

model
lm_head


In [8]:
class LoraLinear(nn.Module):
    def __init__(self, 
                 base_layer: nn.Linear,
                 r: int = 8,
                 alpha: int = 16,
                 dropout_p: float = 0,
                 test_mode = False
                 ) -> None:
        super(LoraLinear, self).__init__()
        self.base_layer = deepcopy(base_layer)
        self.r = r
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout_p)

        self.Lora_A = nn.Parameter(torch.empty((r, base_layer.in_features), dtype=base_layer.weight.dtype))
        self.Lora_B = nn.Parameter(torch.empty((base_layer.out_features, r), dtype=base_layer.weight.dtype))

        nn.init.normal_(self.Lora_A, mean=0.0, std=0.02)

        if test_mode:
            nn.init.normal_(self.Lora_A, mean=0.0, std=0.02)
        else:
            nn.init.zeros_(self.Lora_B)

        for param in self.base_layer.parameters():
            param.requires_grad = False

    def forward(self, x):
        scaling = float(self.alpha) / float(self.r)
        lora_adjustment = F.linear(self.dropout(x), self.Lora_A)
        lora_adjustment = F.linear(lora_adjustment, self.Lora_B)
        return self.base_layer(x) + scaling * lora_adjustment

In [9]:
def linear_model_detect(model_name):
    for s in ['embed', 'norm', 'lm_head']:
        if s in model_name:
            return True, s
    return False, None
    
def replace_linear_with_lora(
    module: nn.Module,
    r: int = 8,
    alpha: int = 16,
    dropout_p = 0.0,
    embed_requires_grad: bool = False,
    normal_requires_grad: bool = False,
    head_requires_grad:bool = False,
    test_mode: bool = False
):
    def linear_model_detect(model_name):
        grad_map = {
            'norm': normal_requires_grad,
            'embed': embed_requires_grad,
            'lm_head': head_requires_grad,
        }
        for s in grad_map.keys():
            if s in model_name:
                return True, grad_map[s]
        return False, False

    for name, child in module.named_children():
        exist, grad_set = linear_model_detect(name)
        if exist: 
            for param in child.parameters():
                param.requires_grad = grad_set
        elif isinstance(child, nn.Linear):
            lora_linear = LoraLinear(child, r=r, alpha=alpha, dropout_p=dropout_p, test_mode=test_mode)
            setattr(module, name, lora_linear)
        else:
            replace_linear_with_lora(
                child, r=r, alpha=alpha, dropout_p=dropout_p,
                embed_requires_grad=embed_requires_grad,
                normal_requires_grad=embed_requires_grad,
                head_requires_grad=head_requires_grad
            )

In [10]:
def print_trainable_parameters(module: nn.Module):
    total_params = sum(p.numel() for p in module.parameters())
    trainable_parameters = sum(p.numel() for p in module.parameters() if p.requires_grad)
    trainable_percentage = trainable_parameters / total_params
    print(f"trainable params: {trainable_parameters:,} || all params: {total_params:,} || trainable%: {trainable_percentage:.4f}")

In [11]:
print_trainable_parameters(model)

trainable params: 4,022,468,096 || all params: 4,022,468,096 || trainable%: 1.0000


In [12]:
lora_model = deepcopy(model)
replace_linear_with_lora(lora_model, r=8, alpha=16)
print_trainable_parameters(lora_model)

: 

In [13]:
print(lora_model)

BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(46145, 2048)
    (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): LoraLinear(
            (base_layer): Linear(in_features=2048, out_features=6144, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (dense): LoraLinear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): LoraLinear(
            (base_layer): Linear(in_features=2048, ou

In [14]:
def print_model_parameters(model):
    """
    打印模型参数
    """
    for name, parameter in model.named_parameters():
        print(f"{name}: | Requires_grad: {parameter.requires_grad}")

In [15]:
print_model_parameters(lora_model)

transformer.word_embeddings.weight: | Requires_grad: False
transformer.word_embeddings_layernorm.weight: | Requires_grad: False
transformer.word_embeddings_layernorm.bias: | Requires_grad: False
transformer.h.0.input_layernorm.weight: | Requires_grad: False
transformer.h.0.input_layernorm.bias: | Requires_grad: False
transformer.h.0.self_attention.query_key_value.Lora_A: | Requires_grad: True
transformer.h.0.self_attention.query_key_value.Lora_B: | Requires_grad: True
transformer.h.0.self_attention.query_key_value.base_layer.weight: | Requires_grad: False
transformer.h.0.self_attention.query_key_value.base_layer.bias: | Requires_grad: False
transformer.h.0.self_attention.dense.Lora_A: | Requires_grad: True
transformer.h.0.self_attention.dense.Lora_B: | Requires_grad: True
transformer.h.0.self_attention.dense.base_layer.weight: | Requires_grad: False
transformer.h.0.self_attention.dense.base_layer.bias: | Requires_grad: False
transformer.h.0.post_attention_layernorm.weight: | Requires_g

In [35]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules='all-linear'
)
peft_lora_model = deepcopy(model)
peft_lora_model = get_peft_model(peft_lora_model, lora_config)
peft_lora_model.print_trainable_parameters()


trainable params: 63,744 || all params: 242,136 || trainable%: 26.3257


In [18]:
peft_lora_model

PeftModel(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 2048)
        (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=6144, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=6144, bias=False)
                )
                (lora_embedding_A): ParameterDict()
   